# 🚀 VS#-15M Deep Neural Language Model Trainer on Kaggle
### Free Accelerated Training on NVIDIA T4 / P100 GPU (30 Hours/Week Free!)

This notebook trains the **VS#-15M** (15 Million Parameters) neural network on Kaggle's free GPUs (Tesla T4 x2 or Tesla P100).

**⚠️ IMPORTANT KAGGLE SETUP:**
1. In the right-hand sidebar under **Settings > Accelerator**, select **GPU T4 x2** (or **GPU P100**).
2. In the right-hand sidebar under **Settings > Internet**, ensure **Internet on** is enabled (required to clone from GitHub).

## 1. Verify GPU Setup
Check your assigned Kaggle GPU (Tesla T4 or P100 with 16GB VRAM).

In [ ]:
!nvidia-smi

## 2. Clone VS-Sharp Repository into `/kaggle/working`

In [ ]:
import os
%cd /kaggle/working
!rm -rf VS-Sharp
!git clone https://github.com/VerScript/VS-Sharp.git
%cd /kaggle/working/VS-Sharp
print('Working directory:', os.getcwd())

## 3. Verify Training Dataset
Inspect the master training dataset (1,142+ rich pairs across VerScript Academy, Standard Libraries, Tests, and Polyglot `inject` languages).

In [ ]:
import json
with open('knowledge/training_data.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)
print(f'✅ Dataset loaded successfully: {len(dataset):,} prompt-response pairs ready for training!')

## 4. Run 15M Parameter GPU Training
Trains on Kaggle GPU Tensor Cores. Evaluates conversational, VerScript, Python, JavaScript, and C test suites after **every single epoch**.

In [ ]:
!python train_gpu.py --epochs 50 --batch-size 128 --samples-per-epoch 15000 --lr 0.001

## 5. Visualize Loss and Accuracy Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open('epoch_eval_history.json', 'r') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
accuracies = [h['accuracy'] * 100 for h in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, losses, 'r-o', label='Cross-Entropy Loss')
plt.title('VS#-15M Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, accuracies, 'b-o', label='Accuracy')
plt.title('VS#-15M Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## 6. Interactive Prompt Tester
Test any custom prompt directly against the newly trained 15M weights in Kaggle.

In [ ]:
import torch
from train_gpu import VSSharp15M, generate_text, calculate_hidden_size

with open('model_weights.json', 'r') as f:
    w_data = json.load(f)

vocab = w_data['vocab']
vocab_map = {t: idx for idx, t in enumerate(vocab)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

hidden_size_1 = calculate_hidden_size(len(vocab))
model = VSSharp15M(len(vocab), hidden_size_1).to(device)

w = w_data['weights']
model.embedding.weight.data.copy_(torch.tensor(w['E'], dtype=torch.float32))
model.fc1.weight.data.copy_(torch.tensor(w['W1'], dtype=torch.float32).t())
model.fc1.bias.data.copy_(torch.tensor(w['b1'], dtype=torch.float32))
model.fc2.weight.data.copy_(torch.tensor(w['W2'], dtype=torch.float32).t())
model.fc2.bias.data.copy_(torch.tensor(w['b2'], dtype=torch.float32))
model.fc_out.weight.data.copy_(torch.tensor(w['W3'], dtype=torch.float32).t())
model.fc_out.bias.data.copy_(torch.tensor(w['b3'], dtype=torch.float32))

test_prompt = "write a hello world program in verscript"
response = generate_text(model, vocab, vocab_map, test_prompt, device, max_tokens=60, temperature=0.7)
print(f"Prompt: {test_prompt}")
print(f"Generated:\n{response}")

## 7. Export Outputs for Kaggle Download
Copies `model_weights.json.gz` (~35 MB) and evaluation history into `/kaggle/working/` so they appear in Kaggle's **Data > Output** panel on the right sidebar.

In [ ]:
!cp /kaggle/working/VS-Sharp/model_weights.json.gz /kaggle/working/
!cp /kaggle/working/VS-Sharp/epoch_eval_history.json /kaggle/working/
print('✅ Copied weights to /kaggle/working/')
print('👉 Look at the right sidebar under Data > Output to download model_weights.json.gz with 1 click!')

## 8. Push Trained Weights Directly to GitHub (Optional)
If you have a GitHub Personal Access Token (PAT), you can push the learning directly back to GitHub:

In [ ]:
import getpass
github_token = getpass.getpass('Enter your GitHub Personal Access Token: ')

!git config --global user.name "VS-Sharp Kaggle Trainer"
!git config --global user.email "trainer@verscript.io"
!git add model_weights.json.gz epoch_eval_history.json knowledge/training_data.json
!git commit -m "feat(weights): push trained 15M model weights from Kaggle GPU"
!git push https://{github_token}@github.com/VerScript/VS-Sharp.git main
print('✅ Successfully pushed model weights to GitHub!')